---
title : "Configuratie Agent"
execute:
    output: true
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

De configuratie Agent van 'Talk with FEWS' heeft verschillende tools tot zijn beschikking.
Het taalmodel kan deze zelf invullen, maar hier wordt per tool een voorbeeld gegeven van de in en output. 

```python
# TODO: 
from talk_with_fews.tools.config.deadpoint_analysis import analyze_dead_points
from talk_with_fews.tools.config.dependency_analysis import analyze_dependencies
from talk_with_fews.tools.config.graph_visualize import visualize_graph
from talk_with_fews.tools.config.workflow_hierarchy import find_workflow_call_hierarchy
```

### Voorbereidingen
In eerste instantie worden de juiste variable opgehaald. Het context object bevat informatie over welke configuratie gebruikt wordt. In de voorbereiding is de configuratie al ingelezen in een database object. Dit is anders dan de normale 'FEWS database' waar alle data in zit. In deze notebook is alles met 'database' dus de configuratie database.


In [1]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

c:\Data\Python\PR5473.10_Talk_with_Fews\talk-with-fews\.pixi\envs\default\Lib\site-packages\pydantic\fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


## Search workflows{#sec-search-workflows}
Om meer inzicht te krijgen in de configuratie van FEWS kan naar verschillende dingen gezocht worden.

De drie zoek mogelijkheden zijn:

- [`search_workflows`](#sec-search-workflows)
- [`search_timeseries`](#sec-search-timeseries)
- [`search_transformations`](#sec-search-transformations)

Workflows is een logisch start punt om de verschillende tools uit te leggen. 
Gegeven een (deel) van de workflow naam kan de informatie achter deze workflow worden opgehaald.
Ik weet dat ik een workflow wil hebben die `im.wiwb.knmi` bevat, ik zoek of er workflows zijn die deze naam bevatten.

In [2]:
name_pattern = "im.wiwb.knmi"

In [3]:
from talk_with_fews.tools.config.workflow_search import search_workflows
from talk_with_fews.models.workflow_search import (
    WorkflowSearchOutput,
    WorkflowSearchInput,
)

workflow_search_input = WorkflowSearchInput(name_pattern=name_pattern)
workflow_search_output: WorkflowSearchOutput = await search_workflows(
    ctx=ctx, input=workflow_search_input
)

In [4]:
# | attr.output='.details summary="Output"'
print(format_as_xml(workflow_search_output, root_tag="WorkflowDetailsOutput"))

<WorkflowDetailsOutput>
  <results>
    <item>
      <id>59</id>
      <wf_id>wf.im.wiwb.knmi</wf_id>
      <desc>Taak om WIWB KNMI metingen te importeren</desc>
      <module_config_ids>
        <item>244</item>
      </module_config_ids>
    </item>
    <item>
      <id>61</id>
      <wf_id>wf.im.wiwb.knmi.eps</wf_id>
      <desc>Taak om WIWB KNMI Regional EPS verwachtingen te importeren</desc>
      <module_config_ids>
        <item>246</item>
      </module_config_ids>
    </item>
    <item>
      <id>60</id>
      <wf_id>wf.im.wiwb.knmi.harmonie</wf_id>
      <desc>Taak om WIWB KNMI Harmonie verwachtingen te importeren</desc>
      <module_config_ids>
        <item>245</item>
      </module_config_ids>
    </item>
  </results>
  <markdown_content>## Workflow Search Results
Found 3 workflows matching the pattern 'im.wiwb.knmi'.

| ID | Workflow ID | Description | Module Config IDs |
|----|-------------|-------------|-------------------|
| 59 | wf.im.wiwb.knmi | Taak om WIWB KNMI me

## Get workflow details
Gegeven een workflow ID (bijvoorbeeld uit `search_workflows`) kan de bijbehorende informatie over deze workflow worden opgehaald.

In [5]:
workflow_id = workflow_search_output.results[0].id
workflow_id

59

In [6]:
from talk_with_fews.tools.config.workflows_details import get_workflow_details
from talk_with_fews.models.workflow_details import (
    WorkflowDetailsOutput,
    WorkflowDetailsInput,
)

workflow_details_input = WorkflowDetailsInput(workflow_id=workflow_id)
workflow_details_output: WorkflowDetailsOutput = await get_workflow_details(
    ctx=ctx, input=workflow_details_input
)

In [7]:
print(format_as_xml(workflow_details_output, root_tag="WorkflowDetailsOutput"))

<WorkflowDetailsOutput>
  <results>
    <item>
      <id>59</id>
      <wf_id>wf.im.wiwb.knmi</wf_id>
      <desc>Taak om WIWB KNMI metingen te importeren</desc>
      <wffn>wf.im.wiwb.knmi</wffn>
      <module_calls>
        <item>
          <mic_id>244</mic_id>
          <modins>im.wiwb.knmi</modins>
          <ord_nr>0</ord_nr>
        </item>
      </module_calls>
    </item>
  </results>
  <markdown_content>## Workflow: wf.im.wiwb.knmi
**Description:** Taak om WIWB KNMI metingen te importeren

### Basic Information
| Property | Value |
|----------|--------|
| ID | 59 |
| Function Name | wf.im.wiwb.knmi |

### Module Calls and Their Transformations

#### Module: im.wiwb.knmi
The following transformations are executed in this module:

| Order | Transformation ID | Description |
|-------|------------------|-------------|
| 1 | 1357 | im.wiwb.knmi |
| 2 | 1358 | im.wiwb.knmi |
| 3 | 1359 | im.wiwb.knmi |
| 4 | 1360 | im.wiwb.knmi |
| 5 | 1361 | im.wiwb.knmi |
| 6 | 1362 | im.wiwb.knmi

## Get transformation details{#sec-get-transformation-details}
Gegeven de transformatie ID kunnen we kijken wat verder met deze workflow te maken heeft. Bijvoorbeeld uit tabel in de `get_workflow_details` tool. 

In [8]:
transformation_id = 1357

In [9]:
from talk_with_fews.tools.config.transformation_details import (
    get_transformation_details,
)
from talk_with_fews.models.transformation_details import (
    TransformationDetailsInput,
    TransformationDetailsOutput,
)

transformation_details_input = TransformationDetailsInput(
    transformation_id=transformation_id
)
transformation_details_output: TransformationDetailsOutput = (
    await get_transformation_details(ctx=ctx, input=transformation_details_input)
)

In [10]:
print(
    format_as_xml(transformation_details_output, root_tag="TransformationDetailsOutput")
)

<TransformationDetailsOutput>
  <results>
    <item>
      <id>1357</id>
      <desc>im.wiwb.knmi</desc>
      <ord_nr>1</ord_nr>
      <config_file>/ModuleConfigFiles/WIWB/im.wiwb.knmi.xml</config_file>
      <mic_id>244</mic_id>
      <modins>im.wiwb.knmi</modins>
      <time_series_links>
        <item>
          <inout>out</inout>
          <tss_id>159</tss_id>
        </item>
      </time_series_links>
    </item>
  </results>
  <markdown_content>## Transformation Details
**ID:** 1357
**Description:** im.wiwb.knmi
**Config File:** /ModuleConfigFiles/WIWB/im.wiwb.knmi.xml

### Basic Information
| Property | Value |
|----------|--------|
| Order Number | 1 |
| Module Instance | im.wiwb.knmi |
| Module Config ID | 244 |

### Time Series Connections
This transformation processes the following time series:

#### Input Time Series
No input time series connected.

#### Output Time Series
| ID | Description |
|----|-------------|
| 159 | Output time series |

### Transformation Flow
```me

## Describe Timeseries
Gegeven een timeseries ID kan je bijbehorende informatie over de transformaties ophalen waar deze timeseries in voor komt. 
Hier gebruiken we een timeseries ID uit [`get_transformation_details`](#sec-get-transformation-details). 

In [11]:
timeseries_id = transformation_details_output.results[0].time_series_links[0].tss_id
timeseries_id

159

In [12]:
from talk_with_fews.tools.config.timeseries_describe import describe_timeseries
from talk_with_fews.models.timeseries_describe import (
    DescribeTimeSeriesInput,
    DescribeTimeSeriesOutput,
)

timeseries_describe_input = DescribeTimeSeriesInput(timeseries_id=timeseries_id)
timeseries_describe_output: DescribeTimeSeriesOutput = await describe_timeseries(
    ctx=ctx, input=timeseries_describe_input
)

In [13]:
print(format_as_xml(timeseries_describe_output, root_tag="DescribeTimeSeriesOutput"))

<DescribeTimeSeriesOutput>
  <results>
    <item>
      <id>159</id>
      <modins>im.wiwb.knmi</modins>
      <vtype>scalar</vtype>
      <parid>P.G.d9</parid>
      <qualifiers />
      <tstype>external historical</tstype>
      <timess />
      <description>im.wiwb.knmi•scalar•P.G.d9••external historical•</description>
      <endpoint>False</endpoint>
      <endpoint_desc>null</endpoint_desc>
      <transformations />
    </item>
  </results>
  <markdown_content>## Time Series 159
**Description:** im.wiwb.knmi•scalar•P.G.d9••external historical•

### Properties
| Property | Value |
|----------|--------|
| Module Instance | im.wiwb.knmi |
| Value Type | scalar |
| Parameter ID | P.G.d9 |
| Qualifiers |  |
| Time Series Type | external historical |
| Time Step |  |

### Transformations
No transformations found</markdown_content>
</DescribeTimeSeriesOutput>


## Search Timeseries{#sec-search-timeseries}
Naast zoeken op workflows en dan via transformaties naar de bijbehorende timeseries, zoals met [`search_workflow`](#sec-search-workflows), kan je ook direct zoeken op timeseries.

Dit kan op basis van:
- parameter_id
- qualifier_pattern
- module_instance
- endpoint
- include_transformations

Hier worden bijbehorende timeseries opgehaald die voldoen aan de opgegeven criteria.

In [14]:
from talk_with_fews.tools.config.timeseries_search import search_timeseries
from talk_with_fews.models.timeseries_search import (
    TimeSeriesSearchInput,
    TimeSeriesSearchOutput,
)

timeseries_search_input = TimeSeriesSearchInput(module_instance="im.wiwb.knmi")

timeseries_search_output: TimeSeriesSearchOutput = await search_timeseries(
    ctx=ctx, input=timeseries_search_input
)

In [15]:
print(format_as_xml(timeseries_search_output, root_tag="TimeSeriesSearchOutput"))

<TimeSeriesSearchOutput>
  <results>
    <item>
      <id>142</id>
      <modins>im.wiwb.knmi</modins>
      <vtype>scalar</vtype>
      <parid>P.G.h</parid>
      <qualifiers />
      <tstype>external historical</tstype>
      <timess>hour</timess>
      <description>im.wiwb.knmi|scalar|P.G.h||external historical|hour</description>
      <endpoint>True</endpoint>
      <endpoint_desc>null</endpoint_desc>
      <transformations>null</transformations>
    </item>
    <item>
      <id>142</id>
      <modins>im.wiwb.knmi</modins>
      <vtype>scalar</vtype>
      <parid>P.G.h</parid>
      <qualifiers />
      <tstype>external historical</tstype>
      <timess>hour</timess>
      <description>im.wiwb.knmi|scalar|P.G.h||external historical|hour</description>
      <endpoint>True</endpoint>
      <endpoint_desc>null</endpoint_desc>
      <transformations>null</transformations>
    </item>
    <item>
      <id>143</id>
      <modins>im.wiwb.knmi</modins>
      <vtype>scalar</vtype>
      <pa

## Search Transformation{#sec-search-transformations}
Als laatste van de searches: [`search_workflows`](#sec-search-workflows),[`search_workflows`](#sec-search-timeseries), kan je ook zoeken op transformaties. Dit is als configurator iets minder intuïtief, maar kan handig zijn voor de AI-agent om door te zoeken naar andere transformaties. 

Dit kan op basis van:
- description_pattern
- workflow_name
- module_config
- max_depth


In [23]:
from talk_with_fews.tools.config.transformation_search import search_transformations
from talk_with_fews.models.transformation_search import (
    TransformationSearchInput,
    TransformationSearchOutput,
)

transformation_search_input = TransformationSearchInput(
    description_pattern="im.wiwb.knmi", workflow_name="wf.im.wiwb.knmi"
)
transformation_search_output: TransformationSearchOutput = await search_transformations(
    ctx=ctx, input=transformation_search_input
)

In [24]:
print(format_as_xml(transformation_search_output, root_tag="TimeSeriesSearchOutput"))

<TimeSeriesSearchOutput>
  <results>
    <item>
      <id>1357</id>
      <desc>im.wiwb.knmi</desc>
      <ord_nr>1</ord_nr>
      <mic_id>244</mic_id>
      <modins>im.wiwb.knmi</modins>
      <workflow_id>59</workflow_id>
      <workflow_name>wf.im.wiwb.knmi</workflow_name>
      <workflow_desc>Taak om WIWB KNMI metingen te importeren</workflow_desc>
    </item>
    <item>
      <id>1358</id>
      <desc>im.wiwb.knmi</desc>
      <ord_nr>2</ord_nr>
      <mic_id>244</mic_id>
      <modins>im.wiwb.knmi</modins>
      <workflow_id>59</workflow_id>
      <workflow_name>wf.im.wiwb.knmi</workflow_name>
      <workflow_desc>Taak om WIWB KNMI metingen te importeren</workflow_desc>
    </item>
    <item>
      <id>1359</id>
      <desc>im.wiwb.knmi</desc>
      <ord_nr>3</ord_nr>
      <mic_id>244</mic_id>
      <modins>im.wiwb.knmi</modins>
      <workflow_id>59</workflow_id>
      <workflow_name>wf.im.wiwb.knmi</workflow_name>
      <workflow_desc>Taak om WIWB KNMI metingen te importeren</

## Get database schema
Om op de juiste manier de tool execute query te gebruiken, is het handig om te weten wat er in de database zit.
Hiervoor zijn de schema's van de configuratie database.
Deze tool geeft een overzicht van de beschikbare tabellen en kolommen.

In [17]:
from talk_with_fews.tools.config.get_database_schema import get_database_schema
from talk_with_fews.models.database_schema import (
    DatabaseSchemaOutput,
    DatabaseSchemaInput,
)

database_schema_input = DatabaseSchemaInput()

database_schema_output: DatabaseSchemaOutput = await get_database_schema(
    ctx=ctx, input=database_schema_input
)

In [18]:
print(format_as_xml(database_schema_output, root_tag="DatabaseSchemaOutput"))

<DatabaseSchemaOutput>
  <tables>
    <call_stacks>
      <item>id</item>
      <item>wf_id</item>
      <item>desc</item>
      <item>wffn</item>
      <item>parent_id</item>
      <item>props_set_id</item>
    </call_stacks>
    <endpoints>
      <item>tss_id</item>
      <item>desc</item>
    </endpoints>
    <links>
      <item>trafo_id</item>
      <item>tss_id</item>
      <item>inout</item>
    </links>
    <modins_calls>
      <item>id</item>
      <item>call_stack_id</item>
      <item>modins</item>
      <item>props_set_id</item>
    </modins_calls>
    <props_sets>
      <item>id</item>
      <item>key</item>
      <item>value</item>
    </props_sets>
    <trafos>
      <item>id</item>
      <item>mic_id</item>
      <item>ord_nr</item>
      <item>desc</item>
      <item>config_file</item>
    </trafos>
    <ts_sets>
      <item>id</item>
      <item>modins</item>
      <item>vtype</item>
      <item>parid</item>
      <item>qualifiers</item>
      <item>tstype</item>
     

## Execute query
Deze tool geeft alle mogelijke vrijheid: het taalmodel kan elke SQL query uitvoeren.

In [19]:
from talk_with_fews.tools.config.sql_query import execute_query
from talk_with_fews.models.sql_query import SQLQueryInput, SQLQueryOutput

sql_query_input = SQLQueryInput(
    query="SELECT * FROM endpoints limit 3",
)

sql_query_output: SQLQueryOutput = await execute_query(ctx=ctx, input=sql_query_input)

In [20]:
print(format_as_xml(sql_query_output, root_tag="SQLQueryOutput"))

<SQLQueryOutput>
  <results>
    <item>
      <tss_id>1</tss_id>
      <desc>filters</desc>
    </item>
    <item>
      <tss_id>2</tss_id>
      <desc>filters</desc>
    </item>
    <item>
      <tss_id>3</tss_id>
      <desc>filters</desc>
    </item>
  </results>
  <column_names>
    <item>tss_id</item>
    <item>desc</item>
  </column_names>
  <row_count>3</row_count>
  <query_executed>SELECT * FROM endpoints limit 3</query_executed>
</SQLQueryOutput>
